# Step 2.2 — Clean and Enrich the Transactions Table
Transforming the FMCG dataset into the Meridian Industrial Supplies context.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../00_raw_data/FMCG Daily Sales Data/FMCG_2022_2024.csv')

# ── 1. Parse dates ──────────────────────────────
df['date'] = pd.to_datetime(df['date'])
df['year']    = df['date'].dt.year
df['month']   = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter
df['week']    = df['date'].dt.isocalendar().week

# ── 2. Rename columns to Meridian context ───────
df.rename(columns={
    'sku': 'product_id',
    'price_unit': 'unit_price_gbp'
}, inplace=True)

# ── 3. Drop duplicate rows ─────────────────────
before = len(df)
df = df.drop_duplicates(subset=['date', 'product_id', 'region', 'channel'])
after = len(df)
print(f"Duplicates removed: {before - after:,}")

# ── 4. Assign UK regions ───────────────────────
np.random.seed(42)
uk_regions = ['North', 'Midlands', 'South East', 'South West', 'London']
df['region'] = np.random.choice(uk_regions, size=len(df))

# ── 5. Assign customer type ────────────────────
types = ['Trade', 'Retail', 'Government', 'Contractor']
df['customer_type'] = np.random.choice(types, size=len(df),
                       p=[0.45, 0.30, 0.15, 0.10])

# ── 6. Create product line hierarchy ───────────
product_line_map = {
    'Milk': 'MRO Supplies',
    'Juice': 'Warehouse Equipment',
    'ReadyMeal': 'Industrial Hardware',
    'SnackBar': 'Safety Products',
    'Yogurt': 'Packaging'
}
df['product_line'] = df['category'].map(product_line_map)

# ── 7. Calculate revenue ───────────────────────
df['revenue_gbp'] = df['units_sold'] * df['unit_price_gbp']

print(f"Cleaning complete: {len(df):,} rows")

In [ ]:
# ── Investigate zero/negative revenue before removing ──
print(f"Total rows: {len(df):,}")
print(f"Zero revenue rows: {(df['revenue_gbp'] == 0).sum():,}")
print(f"Negative revenue rows: {(df['revenue_gbp'] < 0).sum():,}")

print(f"\n── Zero-revenue breakdown by product line ──")
print(df[df['revenue_gbp'] == 0]['product_line'].value_counts())

print(f"\n── Sample of zero/negative revenue rows ──")
print(df[df['revenue_gbp'] <= 0][['date', 'product_id', 'product_line', 'units_sold', 'unit_price_gbp', 'revenue_gbp']].head(10))

In [ ]:
# ── 8. Remove zero/negative revenue rows ───────
before = len(df)
df = df[df['revenue_gbp'] > 0]
after = len(df)
print(f"Zero/negative revenue rows removed: {before - after:,}")

# ── 9. Save cleaned file ──────────────────────
df.to_csv('../01_cleaned_data/transactions_clean.csv', index=False)
print(f"\nSaved {len(df):,} clean rows")
print(f"Columns: {list(df.columns)}")

# Step 2.3 — Prepare the ONS Benchmark Tables
Extracting the relevant monthly series from the ONS Retail Sales Index and Producer Price Index.

In [ ]:
# ── Inspect ONS Retail Sales Index structure ────
rsi_raw = pd.ExcelFile('../00_raw_data/ONS Retail Sales Index/mainreferencetables.xlsx')
print("Sheet names:")
for sheet in rsi_raw.sheet_names:
    print(f"  → {sheet}")

In [ ]:
# ── Inspect Table 1 M (monthly retail sales) ───
rsi = pd.read_excel(
    '../00_raw_data/ONS Retail Sales Index/mainreferencetables.xlsx',
    sheet_name='Table 1 M',
    header=None,
    nrows=15
)
print(rsi.to_string())

In [ ]:
# ── Load ONS Retail Sales Index ─────────────────
rsi = pd.read_excel(
    '../00_raw_data/ONS Retail Sales Index/mainreferencetables.xlsx',
    sheet_name='Table 1 M',
    skiprows=7
)

# First column is the time period
rsi.rename(columns={rsi.columns[0]: 'period'}, inplace=True)

# Keep only the columns we need
rsi = rsi[['period', 
           'All Retailing, Including Automotive Fuel', 
           'Predominantly Non-food Stores']].copy()

# Rename for clarity
rsi.rename(columns={
    'All Retailing, Including Automotive Fuel': 'rsi_all_retail',
    'Predominantly Non-food Stores': 'rsi_non_food'
}, inplace=True)

# Parse the date (format: "2022 Jan")
rsi['period'] = rsi['period'].astype(str)
rsi = rsi[rsi['period'].str.match(r'^\d{4}\s')]  # Keep only rows that start with a year
rsi['date'] = pd.to_datetime(rsi['period'], format='%Y %b')

# Filter to 2022-2024 to match our transaction data
rsi = rsi[(rsi['date'] >= '2022-01-01') & (rsi['date'] <= '2024-12-31')]

# Convert index values to numeric
rsi['rsi_all_retail'] = pd.to_numeric(rsi['rsi_all_retail'], errors='coerce')
rsi['rsi_non_food'] = pd.to_numeric(rsi['rsi_non_food'], errors='coerce')

# Drop the text period column, keep clean date
rsi = rsi[['date', 'rsi_all_retail', 'rsi_non_food']].reset_index(drop=True)

# Save
rsi.to_csv('../01_cleaned_data/ons_rsi_clean.csv', index=False)
print(f"RSI saved: {len(rsi)} rows")
print(f"Date range: {rsi['date'].min()} to {rsi['date'].max()}")
print(rsi.head())

In [ ]:
# Quick check for duplicate dates
print(f"Unique dates: {rsi['date'].nunique()}")
print(f"\nLast 5 rows:")
print(rsi.tail())

In [ ]:
# ── Keep only the first occurrence of each date (index values, not % changes) ──
rsi = rsi.drop_duplicates(subset='date', keep='first').reset_index(drop=True)

# Re-save
rsi.to_csv('../01_cleaned_data/ons_rsi_clean.csv', index=False)
print(f"RSI corrected: {len(rsi)} rows")
print(f"Date range: {rsi['date'].min()} to {rsi['date'].max()}")
print(f"\nFirst 3 rows:")
print(rsi.head(3))
print(f"\nLast 3 rows:")
print(rsi.tail(3))

In [ ]:
# ── Inspect ONS Producer Price Index structure ──
ppi_raw = pd.ExcelFile('../00_raw_data/ONS Producer Price Index/mm22.xlsx')
print("Sheet names:")
for sheet in ppi_raw.sheet_names:
    print(f"  → {sheet}")

In [ ]:
# ── Inspect PPI data sheet structure ────────────
ppi = pd.read_excel(
    '../00_raw_data/ONS Producer Price Index/mm22.xlsx',
    sheet_name='data',
    header=None,
    nrows=15
)
print(ppi.to_string())

In [ ]:
# ── Check PPI dimensions and find relevant columns ──
ppi = pd.read_excel(
    '../00_raw_data/ONS Producer Price Index/mm22.xlsx',
    sheet_name='data',
    header=None
)
print(f"Shape: {ppi.shape}")

# Look at all column titles (row 0)
titles = ppi.iloc[0, 1:].tolist()
print(f"\nTotal columns: {len(titles)}")

# Search for industrial/manufacturing related columns
print("\n── Columns containing 'manufacture' or 'industrial' or 'metal' or 'hardware' ──")
for i, title in enumerate(titles):
    title_str = str(title).lower()
    if any(word in title_str for word in ['manufactur', 'industrial', 'metal', 'hardware', 'fabricat', 'output']):
        print(f"  Column {i+1}: {title}")

# Check date format further down
print(f"\n── Sample dates from column 0 ──")
print(ppi.iloc[7:20, 0].tolist())

In [ ]:
# ── Check when dates become monthly ─────────────
print("── Dates around row 60-80 ──")
print(ppi.iloc[60:80, 0].tolist())

print("\n── Dates around row 100-110 ──")
print(ppi.iloc[100:110, 0].tolist())

# ── Search for OUTPUT PPI columns (what we actually need) ──
print("\n── Columns containing 'OUTPUT' ──")
for i, title in enumerate(titles):
    title_str = str(title).lower()
    if 'output' in title_str and any(word in title_str for word in ['manufactur', 'metal', 'fabricat', 'general', 'all']):
        print(f"  Column {i+1}: {title}")

In [ ]:
# ── Find where monthly data starts ──────────────
for i in range(100, 300):
    val = str(ppi.iloc[i, 0])
    if 'Jan' in val or 'Feb' in val:
        print(f"Row {i}: {val}")
        break

# Show a few rows around that point
print(f"\n── Rows around monthly start ──")
for i in range(max(0, i-3), i+5):
    print(f"Row {i}: {ppi.iloc[i, 0]}")

In [ ]:
# ── Search further for monthly data ─────────────
for i in range(300, 1164):
    val = str(ppi.iloc[i, 0])
    if 'Jan' in val or 'Feb' in val:
        print(f"Monthly data starts at Row {i}: {val}")
        # Show a few rows
        for j in range(i, i+6):
            print(f"  Row {j}: {ppi.iloc[j, 0]}")
        break

In [ ]:
# ── Check date formats at the end of the file ──
print("── Last 20 rows of column 0 ──")
for i in range(1144, 1164):
    print(f"Row {i}: {ppi.iloc[i, 0]}")

print("\n── All unique date formats (sample) ──")
dates = ppi.iloc[7:, 0].dropna().astype(str).unique()
# Show dates that aren't just years or quarters
monthly = [d for d in dates if len(d) > 7 and 'Q' not in d]
print(f"Possible monthly dates: {monthly[:20]}")

In [ ]:
# ── Find where monthly data starts ──────────────
for i in range(7, 1164):
    val = str(ppi.iloc[i, 0])
    if 'JAN' in val:
        print(f"Monthly data starts at Row {i}: {val}")
        break

# ── Now extract the PPI data ────────────────────
# Column 488 = Overall manufactured products (broad benchmark)
# Column 338 = Other fabricated metal products (Industrial Hardware)
# Column 324 = Metal structures and parts (Warehouse Equipment)
# Column 335 = Light metal packaging (Packaging)

# Get column titles for verification
print(f"\nColumn 488: {ppi.iloc[0, 488]}")
print(f"Column 338: {ppi.iloc[0, 338]}")
print(f"Column 324: {ppi.iloc[0, 324]}")
print(f"Column 335: {ppi.iloc[0, 335]}")

In [ ]:
# ── Extract monthly PPI data ────────────────────
ppi_monthly = ppi.iloc[347:, [0, 488, 338, 324, 335]].copy()

# Set column names
ppi_monthly.columns = ['period', 'ppi_manufactured', 'ppi_fabricated_metal', 
                        'ppi_metal_structures', 'ppi_packaging']

# Parse dates (format: "2022 JAN")
ppi_monthly['period'] = ppi_monthly['period'].astype(str)
ppi_monthly = ppi_monthly[ppi_monthly['period'].str.match(r'^\d{4}\s[A-Z]')]
ppi_monthly['date'] = pd.to_datetime(ppi_monthly['period'], format='%Y %b')

# Filter to 2022-2024
ppi_monthly = ppi_monthly[(ppi_monthly['date'] >= '2022-01-01') & 
                           (ppi_monthly['date'] <= '2024-12-31')]

# Convert to numeric
for col in ['ppi_manufactured', 'ppi_fabricated_metal', 'ppi_metal_structures', 'ppi_packaging']:
    ppi_monthly[col] = pd.to_numeric(ppi_monthly[col], errors='coerce')

# Keep clean columns
ppi_monthly = ppi_monthly[['date', 'ppi_manufactured', 'ppi_fabricated_metal', 
                            'ppi_metal_structures', 'ppi_packaging']].reset_index(drop=True)

# Save
ppi_monthly.to_csv('../01_cleaned_data/ons_ppi_clean.csv', index=False)
print(f"PPI saved: {len(ppi_monthly)} rows")
print(f"Date range: {ppi_monthly['date'].min()} to {ppi_monthly['date'].max()}")
print(f"\nFirst 5 rows:")
print(ppi_monthly.head())
print(f"\nLast 3 rows:")
print(ppi_monthly.tail(3))

In [ ]:
# ── Verify cleaned data folder ──────────────────
import os
print("Files in 01_cleaned_data/:")
for f in os.listdir('../01_cleaned_data/'):
    size = os.path.getsize(f'../01_cleaned_data/{f}') / 1024
    print(f"  {f} ({size:.1f} KB)")

In [ ]:
# ── Step 2.5: Inspect Inventory Dataset ─────────
inv1 = pd.read_excel('../00_raw_data/Warehouse Inventory Dataset/Consumables Report - Oct. 2022.xlsx')
print("Consumables Report:")
print(f"Shape: {inv1.shape}")
print(f"Columns: {list(inv1.columns)}")
print(inv1.head())

print("\n" + "=" * 50)

inv2 = pd.read_excel('../00_raw_data/Warehouse Inventory Dataset/Food Report - Oct. 2022.xlsx')
print("\nFood Report:")
print(f"Shape: {inv2.shape}")
print(f"Columns: {list(inv2.columns)}")
print(inv2.head())

In [ ]:
# ── Check Logistics Warehouse Dataset ───────────
import os
path = '../00_raw_data/Logistics Warehouse Dataset (supplement)'
print("Files:")
for f in os.listdir(path):
    print(f"  {f}")
    if f.endswith('.csv'):
        temp = pd.read_csv(f'{path}/{f}')
        print(f"  Shape: {temp.shape}")
        print(f"  Columns: {list(temp.columns)}")
        print(temp.head(3))
    elif f.endswith('.xlsx'):
        temp = pd.read_excel(f'{path}/{f}')
        print(f"  Shape: {temp.shape}")
        print(f"  Columns: {list(temp.columns)}")
        print(temp.head(3))
    print()

In [ ]:
# ── Step 2.6: Clean Inventory Dataset ───────────
inv = pd.read_csv('../00_raw_data/Logistics Warehouse Dataset (supplement)/logistics_dataset.csv')

# Map categories to Meridian product lines
inv_product_map = {
    'Pharma': 'Safety Products',
    'Automotive': 'Warehouse Equipment',
    'Groceries': 'MRO Supplies',
    'Electronics': 'Industrial Hardware',
    'Clothing': 'Packaging'
}

# Check all categories first
print(f"Categories: {inv['category'].unique()}")
print(f"Category counts:\n{inv['category'].value_counts()}")

In [ ]:
# ── Clean and transform inventory data ──────────
inv_product_map = {
    'Pharma': 'Safety Products',
    'Automotive': 'Warehouse Equipment',
    'Groceries': 'MRO Supplies',
    'Electronics': 'Industrial Hardware',
    'Apparel': 'Packaging'
}

inv['product_line'] = inv['category'].map(inv_product_map)

# Rename item_id to product_id for consistency
inv.rename(columns={'item_id': 'product_id'}, inplace=True)

# Calculate days of supply
inv['days_of_supply'] = inv['stock_level'] / inv['daily_demand']

# Create stock status flag (as per execution guide)
inv['stock_status'] = inv.apply(
    lambda row: 'Understock' if row['days_of_supply'] < row['reorder_point'] 
    else ('Overstock' if row['days_of_supply'] > (row['reorder_point'] * 3) 
    else 'Healthy'), axis=1)

# Calculate estimated waste value for overstock items
inv['stock_value'] = inv['stock_level'] * inv['unit_price']
inv['excess_value'] = inv.apply(
    lambda row: (row['stock_level'] - row['reorder_point'] * 3) * row['unit_price'] 
    if row['stock_status'] == 'Overstock' else 0, axis=1)

# Verify
print(f"Shape: {inv.shape}")
print(f"\nStock status distribution:")
print(inv['stock_status'].value_counts())
print(f"\nDays of supply stats:")
print(inv['days_of_supply'].describe())
print(f"\nProduct lines: {inv['product_line'].unique()}")

# Save
inv.to_csv('../01_cleaned_data/inventory_clean.csv', index=False)
print(f"\nSaved {len(inv):,} rows to inventory_clean.csv")